In [ ]:
from nltk.tokenize import sent_tokenize

import ollama

import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, MiniBatchKMeans
from scipy.spatial.distance import cdist, pdist
from sklearn.metrics import pairwise_distances


from boardgames_recsys.data.filtering import filter_df
from boardgames_recsys.text.llm import *
from boardgames_recsys.models.collaborative_filtering import *
from boardgames_recsys.data.matrix import *
from boardgames_recsys.evaluation.ratings import *
from boardgames_recsys.text.filtering import *
from boardgames_recsys.text.lemmatization import *
from boardgames_recsys.text.embeddings import *

from sklearn.feature_extraction.text import CountVectorizer
from treetaggerwrapper import TreeTagger
from nltk.corpus import stopwords
from nltk import word_tokenize
import textwrap
from string import punctuation
from unicodedata import normalize
from unidecode import unidecode
from itertools import product

from surprise import NMF
from surprise import Dataset
from surprise.reader import Reader

sns.set_theme()
%load_ext autoreload
%autoreload 2

In [ ]:
folder = "../database_cleaned"
avis_clean  = pd.read_csv(f"{folder}/avis_clean.csv", index_col=0)
jeux_clean  = pd.read_csv(f"{folder}/jeux_clean.csv", index_col=0)
users       = pd.read_csv(f"{folder}/users.csv", index_col=0)

min_reviews = 10 
rev_filter = filter_df(avis_clean, min_reviews)

rev_filter = rev_filter.assign(index=rev_filter.index)
rev_filter["Length"] = rev_filter["Comment body"].str.split().apply(len)

lemmas = pd.read_csv("../generated_data/Lemmas_VER_cleaned.csv")
corpus = construction_corpus(lemmas, 5000)
lemmas = lemmas[lemmas["Lemma"].isin(corpus)]

comments_lemmatized = lemmas.groupby("Comment line")["Lemma"].apply(" ".join).reset_index()
rev_filter = rev_filter[rev_filter["index"].isin(comments_lemmatized["Comment line"])]
comments_lemmatized = comments_lemmatized.merge(rev_filter[["Game id", "User id", "index"]], left_on="Comment line", right_on="index")

rev_filter, _ = center_score(rev_filter)
users_means = rev_filter[["User id", "Rating"]].groupby("User id").mean().reset_index()

In [ ]:
users_count = rev_filter.groupby("User id")["Game id"].count()
#sns.histplot(users_count, bins=[0, 50, 100, 200, users_count.max()], stat="percent")
counts, values = np.histogram(users_count, bins=[10, 50, 100, 200, users_count.max()])
counts = np.round(counts / np.sum(counts) * 100, 1)

labels = [f"({values[i]}, {values[i+1]})" for i in range(len(values) - 1)]
labels[3] = "> 200"

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
df = pd.DataFrame(data={"Percent of users":counts, "Number of reviews":labels})
sns.barplot(data=df, x="Number of reviews", y="Percent of users", ax=ax, hue="Number of reviews")
ax.set_xlabel("Number of reviews (min, max)")
ax.set_title("Distribution of users by number of reviews")
fig.savefig("../images/users_activity_barplots.svg", bbox_inches="tight", format="svg")

In [ ]:
matrix_ratings, mask_ratings, users_table, games_table = get_matrix_user_game(rev_filter)
cos_dist_matrix = calc_distance_matrix(matrix_ratings, mask_ratings, "cos")
eucl_dist_matrix = calc_distance_matrix(matrix_ratings, mask_ratings, "euclidean")
matrix_ratings

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

mask = (eucl_dist_matrix == 0)
eucl_dist_matrix[mask] = np.nan
eucl_dist_matrix[np.diag_indices(eucl_dist_matrix.shape[0])] = np.nan
cos_dist_matrix[np.diag_indices(cos_dist_matrix.shape[0])] = np.nan

#sns.heatmap(eucl_dist_matrix.toarray()[:500, :500], cmap="viridis_r", ax=ax1)

N = 150
sns.heatmap(eucl_dist_matrix.toarray()[:N, :N], cmap="flare", ax=ax1)
sns.heatmap(cos_dist_matrix[:N, :N], cmap="flare", ax=ax2)

ax1.set_title("Euclidean distance matrix")
ax2.set_title("Cosine distance matrix")

ax1.set_xticklabels([])
ax2.set_xticklabels([])
ax1.set_yticklabels([])
ax2.set_yticklabels([])

ax1.set_ylabel("150 users")
ax2.set_ylabel("150 users")

ax1.set_xlabel("150 users")
ax2.set_xlabel("150 users")

plt.tight_layout()
fig.savefig("../images/matrix_distances.png", bbox_inches="tight", format="png", dpi=150)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
distances_cos = cos_dist_matrix.flatten()
distances_cos = distances_cos[distances_cos > 0]
sns.histplot(distances_cos, binwidth=0.01, stat="percent", ax=ax2)
ax2.set_title("Cosine distances distribution")
ax2.set_xlabel("Distance")

sns.histplot(eucl_dist_matrix.toarray().flatten(), binwidth=0.1, stat="percent", ax=ax1)
ax1.set_title("Euclidean distances distribution")
ax1.set_xlabel("Distance")
fig.savefig("../images/cos_eucl_dist_distr.svg", bbox_inches="tight", format="svg")

# Phrase embeddings with `BAAI/BG3`

### Sentence splitting

In [ ]:
# rev_filter_embed = rev_filter.copy()

# # Unicode normalization
# rev_filter_embed.loc[:, "Comment body"] = rev_filter_embed["Comment body"].apply(lambda row : normalize("NFKC", row))

# # Replace extra caracters that served as a separation
# rev_filter_embed.loc[:, "Comment body"] = rev_filter_embed["Comment body"].str.replace(r"\*{15,}", " ", regex=True)
# rev_filter_embed.loc[:, "Comment body"] = rev_filter_embed["Comment body"].str.replace(r"-{10,}", " ", regex=True)

# # Add space after . or ? or ! for phrases 
# rev_filter_embed.loc[:, "Comment body"] = rev_filter_embed["Comment body"].str.replace(r'([.!?\)])(?=\S)', r'\1 ', regex=True)

# # replace /' by '
# rev_filter_embed.loc[:, "Comment body"] = rev_filter_embed["Comment body"].str.replace(r"\\{1,}'", r"'", regex=True)

# # Delete *** (more that 7 times) 
# rev_filter_embed["Phrases"] = rev_filter_embed["Comment body"].apply(sent_tokenize)
# rev_filter_embed = rev_filter_embed[["Game id", "User id", "index", "Phrases"]]
# rev_filter_embed = rev_filter_embed.explode("Phrases")

# # Delete phrases that contains only regex
# punc_regex = r"^[^\w\s]+$"
# rev_filter_embed = rev_filter_embed[~rev_filter_embed["Phrases"].str.match(punc_regex, na=False)]
# rev_filter_embed["Length"] = rev_filter_embed["Phrases"].str.split().apply(len)
# rev_filter_embed["Length"].sort_values().tail(10)
# rev_filter_embed.shape

### Embeddings

In [ ]:
# # Embed separately 
# long_row = rev_filter_embed[rev_filter_embed["Length"] > 700]
# rev_filter_embed = rev_filter_embed.drop(long_row.index)

# from FlagEmbedding import BGEM3FlagModel

# model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

# phrases = rev_filter_embed["Phrases"].tolist()

# Time spent : 10h37min
# # Batch encode
# encoded = model.encode(phrases, batch_size=16, max_length=800, return_dense=True)

# encoded_long = model.encode(long_row["Phrases"].item(), max_length=2048, return_dense=True)

# embed = rev_filter_embed.assign(Embedding = list(encoded["dense_vecs"]))
# long_row["Embedding"] = [list(encoded_long["dense_vecs"])]
# embed = pd.concat([embed, long_row])
# # embed
# embed.to_parquet("../generated_data/comments_embed.parquet")

# Clustering phrases KMeans

- Positive -> 200 clusters
- Negative -> 250 clusters
- No separation -> 500 clusters

***
- Discard clusters where mean intra distance > 0.8
- ~~Discard clusters that have less than 10 phrases~~ 

### KMeans clustering on all comments (no pos/neg separation)

In [ ]:
comments_embed = pd.read_parquet("../generated_data/comments_embed.parquet")
comments_embed = comments_embed[comments_embed["Phrases"].str.contains(r'[a-zA-Z]', regex=True)]
comments_embed = comments_embed.drop_duplicates(subset="Phrases", keep="first")

In [ ]:
embeds = np.array(comments_embed["Embedding"].tolist())
mean_inertia = []
inter_centers_dist, separability = [], []
nb_clusters = np.arange(100, 1100, 100)[::-1]

for n in nb_clusters:
    print(n)
    kmeans = MiniBatchKMeans(n_clusters=n, batch_size=1024, random_state=42, verbose=0) 
    kmeans.fit(embeds) 

    mean_inertia.append(kmeans.inertia_ / n)
    
    centroids_dist = cdist(kmeans.cluster_centers_, kmeans.cluster_centers_, metric="euclidean")
   
    mean = np.sum(centroids_dist) / (centroids_dist.size - centroids_dist.shape[0])
    inter_centers_dist.append(mean)
    separability.append(np.min(centroids_dist[centroids_dist > 0]))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))

ax1.vlines(500, 100, 500, colors=sns.color_palette("deep")[1], linestyles="dashed")
ax1.plot(nb_clusters, mean_inertia, marker='o')
ax1.set_title("Mean inertia")
ax1.set_xlabel("Number of clusters")
ax1.set_ylabel("Mean inertia")

ax2.vlines(500, 0, 0.7, colors=sns.color_palette("deep")[1], linestyles="dashed")
ax2.plot(nb_clusters, inter_centers_dist, marker='o')
ax1.set_ylim(100, 2550) #ax2.set_ylim(100, 2550)

#ax2.plot(nb_clusters, separability, marker='o', label="Mininum distance")

ax2.set_title("Mean distance between clusters centroids")
ax2.set_xlabel("Number of clusters")
ax2.set_ylabel("Mean distance")

ax2.set_ylim(0.5, 0.85) #ax2.set_ylim(100, 2550)

plt.legend()
plt.tight_layout()
fig.savefig("../images/phrases_kmeans_metrics.svg", format="svg", bbox_inches="tight")

### $500$ Clusters

In [ ]:
all_embeds = np.array(comments_embed["Embedding"].tolist())
kmeans = MiniBatchKMeans(n_clusters=500, batch_size=1024, random_state=42, verbose=0) 
kmeans.fit(all_embeds)

centers = np.take_along_axis(kmeans.cluster_centers_, kmeans.labels_.reshape(-1, 1), axis=0)
distances = np.linalg.norm(all_embeds - centers, axis=1)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 5))
sns.histplot(kmeans.labels_, ax=ax, bins=np.arange(500))
_, counts = np.unique(kmeans.labels_, return_counts=True)
plt.hlines(np.mean(counts), 0, 499, colors=sns.color_palette("deep")[1], linestyles="dashed", label="Mean")
ax.set_title("Number of phrases per cluster")
ax.set_xlabel("Cluster")
ax.set_ylabel("Number of phrases")
ax.legend()
fig.savefig("../images/phrases_count_distribution.svg", bbox_inches="tight", format="svg")

In [ ]:
comments_clusters = comments_embed.assign(Cluster=kmeans.labels_, Distance_centroid=distances)
mean_dist = comments_clusters.groupby("Cluster")["Distance_centroid"].mean().reset_index()
fig, ax = plt.subplots(1, 1, figsize=(15, 5))
sns.histplot(data=mean_dist["Distance_centroid"], ax=ax, bins=50)
ax.vlines(0.7, 0, 65, colors=sns.color_palette("deep")[1], linestyles="dashed", label="Cutoff value")
ax.legend()
ax.set_ylim(0, 65)
ax.set_xlabel("Mean distance to centroid")
ax.set_ylabel("Number of clusters")
ax.set_title("Mean distance to centroid histogram")
fig.savefig("../images/phrases_mean_dist_distribution.svg", bbox_inches="tight", format="svg")

In [ ]:
filtered = counts[counts >= 5]
np.arange(500)[counts > 3000], np.argsort(counts)
# 426#
comments_clusters[comments_clusters["Cluster"].isin([457])]["Phrases"].shape
#print(textwrap.fill(rev_filter[rev_filter["Game id"] == 8532]["Comment body"].iloc[3], width=100))

In [ ]:
comments_clusters = comments_embed.assign(Cluster=kmeans.labels_, Distance_centroid=distances)
print(comments_clusters.shape)
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# count = comments_clusters["Cluster"].value_counts().reset_index()
# sns.barplot(comments_clusters["Cluster"].value_counts(), ax=ax1)
# ax1.set_xticklabels([])

mean_dist = comments_clusters.groupby("Cluster")["Distance_centroid"].mean().reset_index()
# sns.lineplot(data=mean_dist, x="Cluster", y="Distance_centroid")

comments_clusters = comments_clusters[comments_clusters["Cluster"].isin(mean_dist.loc[mean_dist["Distance_centroid"] < 0.7, "Cluster"])]
# #comments_clusters = comments_clusters[comments_clusters["Cluster"].isin(count.loc[count["count"] >= 10, "Cluster"])]

preserved_clusters = np.sort(comments_clusters["Cluster"].unique())
all_centroids = kmeans.cluster_centers_[preserved_clusters]

In [ ]:
# .parquet contains already clean clusters (see above)
#comments_clusters = pd.read_parquet("../generated_data/comments_clusters.parquet") 
#comments_clusters = comments_clusters.drop_duplicates(subset="Phrases", keep="first")

# Quantative evaluation 

In [ ]:
matrix_ratings, mask_ratings, users_table, games_table = get_matrix_user_game(rev_filter)
cos_dist_matrix = calc_distance_matrix(matrix_ratings, mask_ratings, "cos")

users_table = users_table.to_frame().reset_index().rename(columns={"index":"User index"})
games_table = games_table.to_frame().reset_index().rename(columns={"index":"Game index"})

#pos_comments_clusters = pos_comments_clusters.merge(users_table, on="User id").merge(games_table, on="Game id")
#neg_comments_clusters = neg_comments_clusters.merge(users_table, on="User id").merge(games_table, on="Game id")

## Pos / neg separation

### Note : Shannon entropy ponderation
Here, $p_k$ for each cluster is calculated as a percentage of users comments 
- High entropy -> cluster is more specific
- Low entropy -> cluster is more generic

In [ ]:
clusters_weights = cluster_weight_entropy(comments_clusters)

np.random.seed(1)
top_users = rev_filter.groupby("User id")["Game id"].count().sort_values().tail(50).index
#scores = []
for i, user in enumerate(top_users):
    print(i)
    user_score = eval_all_embeddings(user, matrix_ratings, mask_ratings, users_table, games_table, 
                           cos_dist_matrix, 40, comments_clusters, clusters_weights,
                           comments_lemmatized, lemmas, weight_type="entropy")
    
    scores.append(user_score)

In [ ]:
clusters_weights = cluster_weight_entropy(comments_clusters)

np.random.seed(1)
top_users = rev_filter.groupby("User id")["Game id"].count().sort_values().tail(50).index
scores_top = []
for i, user in enumerate(top_users):
    print(i)
    user_score = eval_all_embeddings(user, matrix_ratings, mask_ratings, users_table, games_table, 
                           cos_dist_matrix, 40, comments_clusters, clusters_weights,
                           comments_lemmatized, lemmas, weight_type="entropy")
    scores_top.append(user_score)

In [ ]:
df_uni = pd.DataFrame(data=[lst2 for lst1 in scores_top for lst2 in lst1],
                       columns=["User index", "Similar users", "Random users", "Distant users", "Game id"])
# df_uni = df_uni.melt(id_vars="Game id", value_vars=["Similar users", "Random users", "Distant users"],
#                      var_name="Users type", value_name="List")
# df_uni[['User index', 'Rouge-1', 'Rouge-2', 'Bleu']] = pd.DataFrame(df_uni["List"].to_list(), columns=['User index', 'Rouge-1', 'Rouge-2', 'Bleu'])
# df_uni = df_uni.drop(columns="List")
# #df_uni = df_uni.groupby(["Users type", "User index"]).mean().reset_index()
# df_uni["Type"] = ["200 Random users"] * df_uni.shape[0]
# #df_uni.to_csv("../images/df_top.csv", index=False)
# df_uni

In [ ]:
#pos_clusters_weights = cluster_weight_count(pos_comments_clusters)
#neg_clusters_weights = cluster_weight_count(neg_comments_clusters)
clusters_weights = cluster_weight_entropy(comments_clusters)

np.random.seed(1)
random_users = np.random.choice(rev_filter["User id"].unique(), size=200, replace=False)
scores = []
for user in random_users:
    user_score = eval_all_embeddings(user, matrix_ratings, mask_ratings, users_table, games_table, 
                           cos_dist_matrix, 40, comments_clusters, clusters_weights,
                           comments_lemmatized, lemmas, weight_type="entropy")
    scores.append(user_score)
    print("-------------------------------") 

In [ ]:
df_uni = pd.DataFrame(data=[lst2 for lst1 in scores for lst2 in lst1],
                       columns=["Similar users", "Random users", "Distant users"])
df_uni = df_uni.melt(value_vars=["Similar users", "Random users", "Distant users"],
                     var_name="Users type", value_name="List")
df_uni[['User index', 'Rouge-1', 'Rouge-2', 'Bleu']] = pd.DataFrame(df_uni["List"].to_list(), columns=['User index', 'Rouge-1', 'Rouge-2', 'Bleu'])
df_uni = df_uni.drop(columns="List")
#df_uni = df_uni.groupby(["Users type", "User index"]).mean().reset_index()
df_uni["Type"] = ["200 Random users"] * df_uni.shape[0]
df_uni.to_csv("../images/df_uni.csv", index=False)

In [ ]:
df_uni = pd.read_csv("../images/df_uni.csv")
df_top = pd.read_csv("../images/df_top.csv")[["User index", "Users type", "Rouge-1", "Rouge-2", "Bleu", "Type"]]

# df_uni = df_uni.groupby(["User index", "Users type", "Type"]).mean().reset_index()
# df_top = df_top.groupby(["User index", "Users type", "Type"]).mean().reset_index()

df = pd.concat([df_uni, df_top])
df = df.groupby(["User index", "Users type", "Type"]).mean().reset_index()

df = df.rename(columns={"Users type":"Prediction by", "Rouge-1":"ROUGE-1", "Rouge-2": "ROUGE-2", "Bleu":"BLEU"})
df = df.sort_values(by=["Prediction by", "Type"], ascending=False)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 6))
sns.violinplot(data=df, x="Type", y="ROUGE-1", hue="Prediction by", ax=ax1, cut=0, fill=False)
sns.violinplot(data=df, x="Type", y="ROUGE-2", hue="Prediction by", ax=ax2, cut=0, fill=False)
sns.violinplot(data=df, x="Type", y="BLEU", hue="Prediction by", ax=ax3, cut=0, fill=False)
ax1.set_ylabel("ROUGE-1 (in %)")
ax2.set_ylabel("ROUGE-2 (in %)")
ax3.set_ylabel("BLEU (in %)")


hue_levels = df["Prediction by"].unique()
type_order = df["Type"].unique()
means = df.groupby(["Type", "Prediction by"]).mean().reset_index()

for i, type_val in enumerate(type_order):
    for j, hue_val in enumerate(hue_levels):
        subset = means[(means["Type"] == type_val) & (means["Prediction by"] == hue_val)]
        if j == 0 or j == 1:
            x_pos = i - 0.268 + 0.268 * j
        else:
            x_pos = i - 0.27 + 0.268 * j
        ax1.scatter(x_pos, subset["ROUGE-1"].values[0], marker='o', color="black", label="(mean)", zorder=10)
        ax2.scatter(x_pos, subset["ROUGE-2"].values[0], marker='o', color="black", label="(mean)", zorder=10)
        ax3.scatter(x_pos, subset["BLEU"].values[0], marker='o', color="black",label="(mean)", zorder=10)

ax1.set_xlabel('')
ax2.set_xlabel('')
ax3.set_xlabel('')

for ax in [ax1, ax2, ax3]:
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles)) 
    ax.legend(unique.values(), unique.keys(), title="Prediction by")

fig.suptitle("Distribution of per-user mean ROUGE, BLEU scores", y=0.95)
plt.tight_layout()
fig.savefig("../images/embeds_rouge_bleu.svg", format="svg", bbox_inches="tight")

In [ ]:
df_top = pd.read_csv("../images/df_top.csv")
df_top.sort_values("Bleu", ascending=False).head(30)
#df_top[df_top["Game id"] == 4577]
#user_id = users_table[users_table["User index"] == 902]["User id"].item()
#ev_filter[(rev_filter["User id"] == user_id) & (rev_filter["Game id"] == 8454)]["Comment body"].item()
df_top=df_top.merge(users_table, on="User index").merge(rev_filter[["User id", "Game id", "Comment body"]])
#df_top[df_top["User id"] == 208].sort_values("Game id").head(27)
print(textwrap.fill(df_top[(df_top["User id"] == 201) & (df_top["Game id"] == 10409)]["Comment body"].iloc[0], width=100))
#df_top[df_top["Users type"] == "Distant users"].sort_values(["Game id", "Rouge-2"], ascending=False).head(30)
#df_top

In [ ]:
np.random.seed(2)
game_scores, phrases = eval_all_embeddings(201, matrix_ratings, mask_ratings, users_table, games_table, cos_dist_matrix,
                                           40, comments_clusters, clusters_weights, comments_lemmatized, lemmas, "entropy", 10409)
game_scores, len(phrases)

In [ ]:
for i in range(3):
    phrases_similar = phrases[0].assign(Batch=assign_batch_number(phrases[0], 2300))
    phrases_batched = phrases_similar.groupby("Batch")["Phrases"].apply("\n ".join).tolist()

    response = call_model_by_batch(phrases_batched, "combine_phrases")
    print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))

In [ ]:
phrases_similar = phrases[0].assign(Batch=assign_batch_number(phrases[0], 2300))
phrases_batched = phrases_similar.groupby("Batch")["Phrases"].apply("\n ".join).tolist()

response = call_model_by_batch(phrases_batched, "combine_phrases")
print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))

In [ ]:
phrases_random = phrases[1].assign(Batch=assign_batch_number(phrases[1], 2300))
phrases_batched = phrases_random.groupby("Batch")["Phrases"].apply("\n ".join).tolist()

response = call_model_by_batch(phrases_batched, "combine_phrases")
print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))

In [ ]:
phrases_distant = phrases[2].assign(Batch=assign_batch_number(phrases[2], 2300))
phrases_batched = phrases_distant.groupby("Batch")["Phrases"].apply("\n ".join).tolist()

response = call_model_by_batch(phrases_batched, "combine_phrases")
print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))

In [ ]:
df_top[df_top["User id"] == 208].sort_values("Game id").head(21)


In [ ]:
np.random.seed(1)
df_top = pd.read_csv("../images/df_top.csv")
df_top = df_top.rename(columns={"Users type":"Prediction by", "Rouge-1":"ROUGE-1", "Rouge-2":"ROUGE-2"}).merge(users_table, on="User index")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
top_users_sample = np.random.choice(df_top["User index"].unique(), size=15, replace=False)
df_top = df_top[df_top["User index"].isin(top_users_sample)]

sns.stripplot(data=df_top, x="User id", y="ROUGE-1", hue="Prediction by", jitter=True, dodge=False, ax=ax1)
sns.stripplot(data=df_top, x="User id", y="ROUGE-2", hue="Prediction by", jitter=True, dodge=False, ax=ax2)

ax1.set_ylim(-4, 105)
ax2.set_ylim(-1, 32)

ax1.set_ylabel("ROUGE-1 (in %)")
ax2.set_ylabel("ROUGE-2 (in %)")

fig.suptitle("ROUGEs scores distribution for most active users (15 out of 50)", y=0.95)
plt.tight_layout()

top_users_sample, df_top["User id"].unique()
fig.savefig("../images/stripplot_top_users.svg", format="svg", bbox_inches="tight")

In [ ]:
np.random.seed(1)
df_top = pd.read_csv("../images/df_uni.csv")
df_top = df_top.rename(columns={"Users type":"Prediction by", "Rouge-1":"ROUGE-1", "Rouge-2":"ROUGE-2"}).merge(users_table, on="User index")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
top_users_sample = np.random.choice(df_top["User index"].unique(), size=15, replace=False)
df_top = df_top[df_top["User index"].isin(top_users_sample)]

sns.stripplot(data=df_top, x="User id", y="ROUGE-1", hue="Prediction by", jitter=True, dodge=False, ax=ax1)
sns.stripplot(data=df_top, x="User id", y="ROUGE-2", hue="Prediction by", jitter=True, dodge=False, ax=ax2)

ax1.set_ylim(-4, 105)
ax2.set_ylim(-1, 32)

ax1.set_ylabel("ROUGE-1 (in %)")
ax2.set_ylabel("ROUGE-2 (in %)")

fig.suptitle("ROUGEs scores distribution for random users (15 out of 200)", y=0.95)
plt.tight_layout()
df_top["User id"].unique()
fig.savefig("../images/stripplot_random_users.svg", format="svg", bbox_inches="tight")

In [ ]:
np.random.seed(1)
fig, ax = plt.subplots(1, 1, figsize=(15, 6))
random_users_sample = np.random.choice(df_uni["User index"].unique(), size=15, replace=False)
df_uni = df_uni[df_uni["User index"].isin(random_users_sample)]

sns.stripplot(data=df_uni, x="User index", y="Rouge-1", hue="Users type", jitter=True)
plt.tight_layout()

In [ ]:
df[df["Rouge-1"] >80]

In [ ]:
users_table[users_table["User index"] == 362]
rev_filter[(rev_filter["User id"] == 934) & (rev_filter["Game id"] == 8452)]["Comment body"].item()

In [ ]:
df_uni = pd.DataFrame(data=[lst2 for lst1 in scores_top for lst2 in lst1],
                       columns=["Similar users", "Random users", "Distant users"])
df_uni = df_uni.melt(value_vars=["Similar users", "Random users", "Distant users"],
                     var_name="Users type", value_name="List")
df_uni[['User index', 'Rouge-1', 'Rouge-2', 'Bleu', 'Game id']] = pd.DataFrame(df_uni["List"].to_list(), columns=['User index', 'Rouge-1', 'Rouge-2', 'Bleu', 'Game id'])
df_uni = df_uni.drop(columns="List")
df_uni["Type"] = ["50 Most active users"] * df_uni.shape[0]
df_uni.to_csv("../images/df_top.csv")

In [ ]:
df_top_all = pd.read_csv("../images/df_random_all.csv")
palette = sns.color_palette("deep")

fig, ax1 = plt.subplots(1, 1, figsize=(15, 6))


np.random.seed(0)
users
users = np.random.choice(df_top_all["User index"].unique(), size=20, replace=False)
df_top_all = df_top_all[df_top_all["User index"].isin(users)]


sns.pointplot(data=df_top_all[df_top_all["Users type"] == "Similar users"], x="User index", y="Rouge-1", ax=ax1, errorbar=None,
              color='black', alpha=0.5, label="Mean")
sns.stripplot(data=df_top_all[df_top_all["Users type"] == "Similar users"], x="User index", y="Rouge-1", ax=ax1, color=palette[1], label="ROUGE-1 per game")

handles, labels = ax1.get_legend_handles_labels()
unique = dict(zip(labels, handles))  # keeps first occurrence
ax1.legend(unique.values(), unique.keys())
ax1.get_xaxis().set_visible(False)
ax1.set_xlabel("Users")
ax1.set_ylabel("ROUGE-1 (in %)")

In [ ]:
# df_uni = pd.DataFrame(data=[lst2 for lst1 in scores for lst2 in lst1],
#                        columns=["Similar users", "Random users", "Distant users"])
# df_uni = df_uni.melt(value_vars=["Similar users", "Random users", "Distant users"],
#                      var_name="Users type", value_name="List")
# df_uni[['User index', 'Rouge-1', 'Rouge-2', 'Bleu']] = pd.DataFrame(df_uni["List"].to_list(), columns=['User index', 'Rouge-1', 'Rouge-2', 'Bleu'])
# df_uni = df_uni.drop(columns="List")
# df_uni = df_uni.groupby(["Users type", "User index"]).mean().reset_index()
# df_uni["Type"] = ["200 Random users"] * df_uni.shape[0]


df_uni = pd.read_csv("../images/df_uni.csv")
df_top = pd.read_csv("../images/df_top.csv")
df = pd.concat([df_uni, df_top])
df

sns.violinplot(data=df, x="Type", y="Rouge-1", hue="Users type", cut=0)

In [ ]:
sns.violinplot(data=df, x="Type", y="Rouge-2", hue="Users type", cut=0)

In [ ]:
# User index = 1496 -> outlier
df_uni[df_uni["Rouge-1"] == df_uni["Rouge-1"].max()], df_uni[df_uni["Rouge-2"] == df_uni["Rouge-2"].max()], df_uni[df_uni["Bleu"] == df_uni["Bleu"].max()]

In [ ]:
df_uni.sort_values("Bleu", ascending=False).head(100)["Users type"].value_counts()

In [ ]:
df_uni[df_uni["User index"] == 1496]

In [ ]:

rev_filter[rev_filter["User id"].isin(users_table[users_table["User index"] == 1496]["User id"])]

In [ ]:
#pos_clusters_weights = cluster_weight_count(pos_comments_clusters)
#neg_clusters_weights = cluster_weight_count(neg_comments_clusters)
clusters_weights = cluster_weight_entropy(comments_clusters)

np.random.seed(1)
random_users = np.random.choice(rev_filter["User id"].unique(), size=200, replace=False)
bigrams, unigrams = [], [] 

for user in random_users:
    u, b = eval_all_embeddings(user, matrix_ratings, mask_ratings, users_table, games_table, 
                           cos_dist_matrix, 40, comments_clusters, clusters_weights,
                           comments_lemmatized, lemmas, weight_type="entropy")
    bigrams.append(b)
    unigrams.append(u)
    print("-------------------------------") 

In [ ]:
bigrams

In [ ]:
df_big = pd.DataFrame(data=[lst2 for lst1 in bigrams for lst2 in lst1],
                       columns=["User index", "Similar users", "Random users", "Distant users"])
df_big = df_big.groupby("User index").mean().reset_index()
df_big = df_big.melt(id_vars=["User index"], value_vars=["Similar users", "Random users", "Distant users"],
                     var_name="Users type", value_name="ROUGE score (in %)")
ax = sns.violinplot(data=df_big, x="Users type", y="ROUGE score (in %)", cut=0)
# ax.set_ylim(0, 20)
df_big

In [ ]:
games_table[games_table["Game index"] == 2466] # 9792
comments_lemmatized[(comments_lemmatized["User id"] == 2303) & (comments_lemmatized["Game id"] == 9792)]

In [ ]:
users.tail(70)

In [ ]:
np.mean(bigrams[0], axis=0), np.mean(unigrams[0], axis=0)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 7), dpi=150)
folder = "../generated_data/embeds_eval"
im00, im10 = plt.imread(f"{folder}/entropy/pos_neg_top.png"), plt.imread(f"{folder}/entropy/pos_neg_random.png")
im01, im11 = plt.imread(f"{folder}/count/pos_neg_top.png"), plt.imread(f"{folder}/count/pos_neg_random.png")
images = [im00, im01, im10, im11]

for i, j in product(range(2), repeat=2):
    print(i, j)
    axes[i, j].imshow(images[i * 2 + j])
    axes[i, j].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    axes[i, j].grid(False)
plt.tight_layout()

axes[0, 0].set_title("Entropy ponderation")
axes[0, 0].set_ylabel("Top users")

axes[0, 1].set_title("Count ponderation")

axes[1, 0].set_ylabel("Random users")

## No pos/neg separation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 7), dpi=150)
folder = "../generated_data/embeds_eval"
im00, im10 = plt.imread(f"{folder}/entropy/all_top.png"), plt.imread(f"{folder}/entropy/all_random.png")
im11 = plt.imread(f"{folder}/count/all_random.png")
images = [im00, im10, im11]

for i, j in product(range(2), repeat=2):
    if i == 0 and j == 1:
        axes[i, j].set_axis_off()
    else:
        axes[i, j].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    axes[i, j].grid(False)

axes[0, 0].imshow(images[0])
axes[1, 0].imshow(images[1])
axes[1, 1].imshow(images[2])
plt.tight_layout()

axes[0, 0].set_title("Entropy ponderation")
axes[0, 1].set_title("Count ponderation")

# Qualitative analysis

## Pos/neg separation

**User 0, Game 9146**

In [ ]:
print(textwrap.fill(rev_filter[(rev_filter["User id"] == 0) & (rev_filter["Game id"] == 9146)]["Comment body"].item(), width=100))

**All $3$ similar clusters**

In [ ]:
pos_clusters_weights = cluster_weight_entropy(pos_comments_clusters)
neg_clusters_weights = cluster_weight_entropy(neg_comments_clusters)

user, game = 0, 9146
phrases = eval_embeddings(user, matrix_ratings, mask_ratings, 
                                users_table, games_table, cos_dist_matrix, 
                                40, users_means, pos_comments_clusters, neg_comments_clusters,
                                pos_clusters_weights, neg_clusters_weights, pod_centroids, neg_centroids,
                                comments_lemmatized, lemmas, specific_game=game, weight_type="entropy")

phrases = phrases.assign(Batch=assign_batch_number(phrases, 2300))
phrases_batched = phrases.groupby("Batch")["Phrases"].apply("\n ".join).tolist()
response = call_model_by_batch(phrases_batched, "combine_phrases")
print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))

**User 1, Game 3370**

In [ ]:
print(textwrap.fill(rev_filter[(rev_filter["User id"] == 1) & (rev_filter["Game id"] == 8444)]["Comment body"].item(), width=100))

**Pos/neg**

In [ ]:
pos_clusters_weights = cluster_weight_entropy(pos_comments_clusters)
neg_clusters_weights = cluster_weight_entropy(neg_comments_clusters)

user, game = 1, 8444
phrases = eval_embeddings(user, matrix_ratings, mask_ratings, 
                                users_table, games_table, cos_dist_matrix, 
                                40, users_means, pos_comments_clusters, neg_comments_clusters,
                                pos_clusters_weights, neg_clusters_weights, pod_centroids, neg_centroids,
                                comments_lemmatized, lemmas, specific_game=game, weight_type="entropy")

print("Nb of selected phrases", phrases.shape[0])

phrases = phrases.assign(Batch=assign_batch_number(phrases, 2300))
phrases_batched = phrases.groupby("Batch")["Phrases"].apply("\n ".join).tolist()
response = call_model_by_batch(phrases_batched, "combine_phrases")
print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))

**No pos/neg, count**

In [ ]:
clusters_weights = cluster_weight_count(comments_clusters)

user, game = 1, 8444
phrases = eval_embeddings(user, matrix_ratings, mask_ratings, 
                                users_table, games_table, cos_dist_matrix, 
                                40, users_means, comments_clusters, comments_clusters,
                                clusters_weights, clusters_weights, all_centroids, all_centroids,
                                comments_lemmatized, lemmas, specific_game=game, weight_type="count")

print("Nb of selected phrases", phrases.shape[0])

phrases = phrases.assign(Batch=assign_batch_number(phrases, 2300))
phrases_batched = phrases.groupby("Batch")["Phrases"].apply("\n ".join).tolist()
response = call_model_by_batch(phrases_batched, "combine_phrases")
print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))

**User 0, Game 9146**

In [ ]:
print(textwrap.fill(rev_filter[(rev_filter["User id"] == 0) & (rev_filter["Game id"] == 9146)]["Comment body"].item(), width=100))

**Choose clusters based on entropy 50/50, pos/neg**

In [ ]:
pos_clusters_weights = cluster_weight_entropy(pos_comments_clusters)
neg_clusters_weights = cluster_weight_entropy(neg_comments_clusters)

user, game = 0, 9146
phrases = eval_embeddings(user, matrix_ratings, mask_ratings, 
                                users_table, games_table, cos_dist_matrix, 
                                40, users_means, pos_comments_clusters, neg_comments_clusters,
                                pos_clusters_weights, neg_clusters_weights, pod_centroids, neg_centroids,
                                comments_lemmatized, lemmas, specific_game=game, weight_type="entropy")

print("Nb of selected phrases", phrases.shape[0])

phrases = phrases.assign(Batch=assign_batch_number(phrases, 2300))
phrases_batched = phrases.groupby("Batch")["Phrases"].apply("\n ".join).tolist()
response = call_model_by_batch(phrases_batched, "combine_phrases")
print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))

**Choose clusters based on count 50/50, pos/neg**

In [ ]:
pos_clusters_weights = cluster_weight_count(pos_comments_clusters)
neg_clusters_weights = cluster_weight_count(neg_comments_clusters)

user, game = 0, 9146
phrases = eval_embeddings(user, matrix_ratings, mask_ratings, 
                                users_table, games_table, cos_dist_matrix, 
                                40, users_means, pos_comments_clusters, neg_comments_clusters,
                                pos_clusters_weights, neg_clusters_weights, pod_centroids, neg_centroids,
                                comments_lemmatized, lemmas, specific_game=game, weight_type="count")

print("Nb of selected phrases", phrases.shape[0])

phrases = phrases.assign(Batch=assign_batch_number(phrases, 2300))
phrases_batched = phrases.groupby("Batch")["Phrases"].apply("\n ".join).tolist()
response = call_model_by_batch(phrases_batched, "combine_phrases")
print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))

**Choose clusters based on count 50/50, no pos/neg**

In [ ]:
clusters_weights = cluster_weight_count(comments_clusters)

user, game = 0, 9146
phrases = eval_embeddings(user, matrix_ratings, mask_ratings, 
                                users_table, games_table, cos_dist_matrix, 
                                40, users_means, comments_clusters, comments_clusters,
                                clusters_weights, clusters_weights, all_centroids, all_centroids,
                                comments_lemmatized, lemmas, specific_game=game, weight_type="count")

print("Nb of selected phrases", phrases.shape[0])

phrases = phrases.assign(Batch=assign_batch_number(phrases, 2300))
phrases_batched = phrases.groupby("Batch")["Phrases"].apply("\n ".join).tolist()
response = call_model_by_batch(phrases_batched, "combine_phrases")
print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))